In [190]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

N_INTERACTIONS = 250_000

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Random seed:", SEED)
print("Interactions to generate:", N_INTERACTIONS)

NumPy version: 2.1.1
Pandas version: 2.2.3
Random seed: 42
Interactions to generate: 250000


In [191]:
RAW_DATA_PATH = Path("../data/raw")

users = pd.read_csv(
    RAW_DATA_PATH / "users.csv",
    parse_dates=["signup_date"]
)

creators = pd.read_csv(
    RAW_DATA_PATH / "creators.csv",
    parse_dates=["signup_date"]
)

content = pd.read_csv(
    RAW_DATA_PATH / "content.csv",
    parse_dates=["created_at"]
)

print("Users:", users.shape)
print("Creators:", creators.shape)
print("Content:", content.shape)

Users: (5000, 7)
Creators: (500, 5)
Content: (10000, 9)


In [192]:
interactions.to_csv(
    "../data/raw/interactions.csv",
    index=False
)

print("Saved corrected interactions.csv")

Saved corrected interactions.csv


In [193]:
content_base = content.merge(
    creators[["creator_id", "followers"]],
    on="creator_id",
    how="left"
)

content_base.head()


,content_id,creator_id,content_type,genre,created_at,duration,tags,is_template,is_recreation,followers
0,CT000001,C0131,mini,Mystery,2025-12-17 12:34:36,24.6,"visual,cinematic",False,False,715
1,CT000002,C0484,mini,Horror,2026-08-01 20:42:55,21.6,"story,ai,animation",False,False,291
2,CT000003,C0195,mini,Sci-Fi,2026-06-27 15:47:40,14.7,ai,False,False,33
3,CT000004,C0328,mini,Romance,2026-04-26 06:39:06,6.6,"ai,cinematic,viral",False,False,49
4,CT000005,C0406,image,Action,2025-11-02 17:31:50,88.7,"trending,visual,story",True,False,74


In [194]:
user_activity = rng.lognormal(
    mean=0.0,
    sigma=0.8,
    size=len(users)
)

user_activity = user_activity / user_activity.sum()

interaction_user_idx = rng.choice(
    len(users),
    size=N_INTERACTIONS,
    p=user_activity
)

interaction_users = users.iloc[
    interaction_user_idx
].reset_index(drop=True)

interaction_users.head()

,user_id,country,age_group,signup_date,following_count,creator_flag,preferred_genres
0,U01544,Canada,45+,2024-06-15 01:28:53,22,False,"['Action', 'Drama', 'Romance']"
1,U02217,Canada,18-24,2024-03-20 21:13:15,27,False,['Sci-Fi']
2,U02802,Canada,25-34,2024-06-27 02:16:45,24,False,"['Fantasy', 'Action', 'Romance', 'Documentary']"
3,U02792,United States,25-34,2024-06-14 03:34:11,24,False,"['Sci-Fi', 'Fantasy']"
4,U00227,Germany,35-44,2024-07-16 13:42:08,25,False,['Romance']


In [195]:
content_popularity = np.log1p(content_base["followers"].values)

content_popularity = content_popularity / content_popularity.sum()

interaction_content_idx = rng.choice(
    len(content_base),
    size=N_INTERACTIONS,
    p=content_popularity
)

interaction_content = content_base.iloc[
    interaction_content_idx
].reset_index(drop=True)

interaction_content.head()

,content_id,creator_id,content_type,genre,created_at,duration,tags,is_template,is_recreation,followers
0,CT009766,C0161,video,Horror,2026-04-19 11:34:47,41.6,"story,cinematic,ai",False,False,120
1,CT000222,C0282,mini,Sci-Fi,2025-10-29 22:59:01,5.5,trending,True,False,68
2,CT005101,C0142,image,Action,2026-04-04 20:13:39,160.2,"character,viral",True,False,40
3,CT003164,C0029,mini,Documentary,2026-02-19 17:06:10,47.8,"creative,animation,cinematic",False,False,91
4,CT006055,C0400,image,Drama,2025-11-29 18:57:20,30.2,cinematic,False,False,18


In [196]:
interactions = pd.DataFrame({
    "user_id": interaction_users["user_id"].values,
    "content_id": interaction_content["content_id"].values,
    "creator_id": interaction_content["creator_id"].values,
    "genre": interaction_content["genre"].values,
    "content_type": interaction_content["content_type"].values,
    "duration": interaction_content["duration"].values,
    "creator_followers": interaction_content["followers"].values,
    "content_created_at": interaction_content["created_at"].values
})

interactions.head()

,user_id,content_id,creator_id,genre,content_type,duration,creator_followers,content_created_at
0,U01544,CT009766,C0161,Horror,video,41.6,120,2026-04-19 11:34:47
1,U02217,CT000222,C0282,Sci-Fi,mini,5.5,68,2025-10-29 22:59:01
2,U02802,CT005101,C0142,Action,image,160.2,40,2026-04-04 20:13:39
3,U02792,CT003164,C0029,Documentary,mini,47.8,91,2026-02-19 17:06:10
4,U00227,CT006055,C0400,Drama,image,30.2,18,2025-11-29 18:57:20


In [197]:
user_preferences = (
    users[["user_id", "preferred_genres"]]
    .copy()
)

user_preferences.head()

,user_id,preferred_genres
0,U00001,"['Action', 'Romance', 'Documentary']"
1,U00002,"['Animation', 'Documentary', 'Comedy', 'Horror']"
2,U00003,"['Documentary', 'Animation', 'Comedy', 'Horror']"
3,U00004,"['Animation', 'Drama']"
4,U00005,"['Romance', 'Mystery']"


In [198]:
import ast

user_preferences["preferred_genres"] = (
    user_preferences["preferred_genres"]
    .apply(ast.literal_eval)
)

interactions = interactions.merge(
    user_preferences,
    on="user_id",
    how="left"
)

interactions["genre_match"] = interactions.apply(
    lambda row: row["genre"] in row["preferred_genres"],
    axis=1
)

interactions["genre_match"].mean()

np.float64(0.249544)

In [199]:
# Generate interaction timestamps only after content creation

interaction_end = pd.Timestamp("2026-08-31")

# Convert content creation timestamps to Unix seconds
content_created_seconds = (
    interactions["content_created_at"].astype("int64") // 10**9
)

interaction_end_seconds = interaction_end.value // 10**9

# Generate a random timestamp between content creation
# and the end of the observation period for each interaction
random_fractions = rng.random(len(interactions))

interaction_seconds = (
    content_created_seconds
    + (
        (interaction_end_seconds - content_created_seconds)
        * random_fractions
    ).astype("int64")
)

interactions["timestamp"] = pd.to_datetime(
    interaction_seconds,
    unit="s"
)

# Calculate content age at the time of interaction
interactions["content_age_days"] = (
    interactions["timestamp"]
    - interactions["content_created_at"]
).dt.total_seconds() / (24 * 60 * 60)

# Calculate freshness
interactions["freshness"] = np.exp(
    -interactions["content_age_days"] / 30
)

interactions[
    [
        "timestamp",
        "content_created_at",
        "content_age_days",
        "freshness"
    ]
].head()

,timestamp,content_created_at,content_age_days,freshness
0,2026-08-12 10:33:38,2026-04-19 11:34:47,114.957535,0.021668
1,2026-04-18 01:23:33,2025-10-29 22:59:01,170.100370,0.003448
2,2026-08-27 11:04:55,2026-04-04 20:13:39,144.618935,0.008062
3,2026-08-02 12:22:59,2026-02-19 17:06:10,163.803345,0.004253
4,2026-06-13 06:20:33,2025-11-29 18:57:20,195.474456,0.001480


In [200]:
print("Interactions before content creation:",
      (interactions["timestamp"] < interactions["content_created_at"]).sum())

print("\nMinimum content age:",
      interactions["content_age_days"].min())

print("Maximum content age:",
      interactions["content_age_days"].max())

print("\nInteraction period:")
print("Start:", interactions["timestamp"].min())
print("End:", interactions["timestamp"].max())

Interactions before content creation: 0

Minimum content age: 0.0001388888888888889
Maximum content age: 455.19275462962963

Interaction period:
Start: 2025-06-01 23:55:16
End: 2026-08-30 23:59:46


In [201]:
interactions.to_csv(
    "../data/raw/interactions.csv",
    index=False
)

print("Saved corrected interactions.csv")

Saved corrected interactions.csv


In [202]:
user_activity_df = pd.DataFrame({
    "user_id": users["user_id"],
    "activity_score": rng.lognormal(
        mean=0,
        sigma=0.7,
        size=len(users)
    )
})

user_activity_df["activity_score"] = (
    user_activity_df["activity_score"] /
    user_activity_df["activity_score"].median()
)

interactions = interactions.merge(
    user_activity_df,
    on="user_id",
    how="left"
)

interactions["activity_score"].describe()

count    250000.000000
mean          1.262878
std           1.007298
min           0.071523
25%           0.614439
50%           0.995412
75%           1.614259
max          17.796978
Name: activity_score, dtype: float64

In [203]:
log_followers = np.log1p(
    interactions["creator_followers"]
)

log_followers = (
    log_followers / log_followers.median()
)

log_duration = np.log1p(
    interactions["duration"]
)

duration_signal = np.exp(
    -((log_duration - np.log(30)) ** 2) / 1.5
)

In [204]:
engagement_score = (
    -1.8
    + 1.0 * interactions["genre_match"].astype(float)
    + 0.45 * interactions["freshness"]
    + 0.25 * log_followers
    + 0.35 * interactions["activity_score"].clip(upper=4)
    + 0.20 * duration_signal
    + rng.normal(0, 0.45, N_INTERACTIONS)
)

In [205]:
engagement_probability = 1 / (
    1 + np.exp(-engagement_score)
)

interactions["engagement_probability"] = (
    engagement_probability.clip(0.01, 0.95)
)

interactions["engagement_probability"].describe()

count    250000.000000
mean          0.359240
std           0.153026
min           0.039999
25%           0.241250
50%           0.331477
75%           0.457557
max           0.934161
Name: engagement_probability, dtype: float64

In [206]:
interactions["impression"] = 1

interactions["clicked"] = (
    rng.random(N_INTERACTIONS)
    < interactions["engagement_probability"]
).astype(int)

In [207]:
interactions["clicked"].mean()

np.float64(0.35902)

In [208]:
watch_fraction = rng.beta(
    a=5,
    b=2,
    size=N_INTERACTIONS
)

interactions["watch_time"] = np.where(
    interactions["clicked"] == 1,
    interactions["duration"] * watch_fraction,
    0
)

interactions["watch_time"] = (
    interactions["watch_time"].round(2)
)

In [209]:
interactions["completion_rate"] = np.where(
    interactions["duration"] > 0,
    interactions["watch_time"] / interactions["duration"],
    0
)

interactions["completion_rate"] = (
    interactions["completion_rate"].clip(0, 1)
)

In [210]:
like_probability = (
    0.02
    + 0.25 * interactions["completion_rate"]
    + 0.10 * interactions["genre_match"].astype(float)
)

like_probability = (
    like_probability
    .clip(0.01, 0.60)
)

interactions["liked"] = (
    (
        rng.random(N_INTERACTIONS)
        < like_probability
    )
    & (interactions["clicked"] == 1)
).astype(int)

In [211]:
save_probability = (
    0.01
    + 0.12 * interactions["completion_rate"]
    + 0.08 * interactions["genre_match"].astype(float)
)

save_probability = save_probability.clip(0.005, 0.35)

interactions["saved"] = (
    (
        rng.random(N_INTERACTIONS)
        < save_probability
    )
    & (interactions["clicked"] == 1)
).astype(int)

In [212]:
share_probability = (
    0.005
    + 0.08 * interactions["completion_rate"]
    + 0.05 * interactions["genre_match"].astype(float)
)

share_probability = share_probability.clip(0.002, 0.20)

interactions["shared"] = (
    (
        rng.random(N_INTERACTIONS)
        < share_probability
    )
    & (interactions["clicked"] == 1)
).astype(int)

In [213]:
comment_probability = (
    0.005
    + 0.05 * interactions["completion_rate"]
)

comment_probability = comment_probability.clip(0.002, 0.15)

interactions["commented"] = (
    (
        rng.random(N_INTERACTIONS)
        < comment_probability
    )
    & (interactions["clicked"] == 1)
).astype(int)

In [214]:
recreate_probability = (
    0.003
    + 0.04 * interactions["completion_rate"]
    + 0.03 * interactions["genre_match"].astype(float)
)

recreate_probability = (
    recreate_probability.clip(0.001, 0.12)
)

interactions["recreated"] = (
    (
        rng.random(N_INTERACTIONS)
        < recreate_probability
    )
    & (interactions["clicked"] == 1)
).astype(int)

In [215]:
interactions["meaningful_engagement"] = (
    (
        (interactions["completion_rate"] >= 0.30)
        |
        (interactions["liked"] == 1)
        |
        (interactions["saved"] == 1)
        |
        (interactions["shared"] == 1)
        |
        (interactions["commented"] == 1)
        |
        (interactions["recreated"] == 1)
    )
    & (interactions["clicked"] == 1)
).astype(int)

In [216]:
print(
    "Meaningful engagement rate:",
    interactions["meaningful_engagement"].mean()
)

Meaningful engagement rate: 0.355924


In [217]:
print("Shape:", interactions.shape)

print("\nColumns:")
print(interactions.columns.tolist())

print("\nEngagement rates:")
print(
    interactions[
        [
            "clicked",
            "liked",
            "saved",
            "shared",
            "commented",
            "recreated",
            "meaningful_engagement"
        ]
    ].mean()
)

Shape: (250000, 25)

Columns:
['user_id', 'content_id', 'creator_id', 'genre', 'content_type', 'duration', 'creator_followers', 'content_created_at', 'preferred_genres', 'genre_match', 'timestamp', 'content_age_days', 'freshness', 'activity_score', 'engagement_probability', 'impression', 'clicked', 'watch_time', 'completion_rate', 'liked', 'saved', 'shared', 'commented', 'recreated', 'meaningful_engagement']

Engagement rates:
clicked                  0.359020
liked                    0.084624
saved                    0.045196
shared                   0.028640
commented                0.014420
recreated                0.015420
meaningful_engagement    0.355924
dtype: float64


In [218]:
interactions.head()

,user_id,content_id,creator_id,genre,content_type,duration,creator_followers,content_created_at,preferred_genres,genre_match,...,impression,clicked,watch_time,completion_rate,liked,saved,shared,commented,recreated,meaningful_engagement
0,U01544,CT009766,C0161,Horror,video,41.6,120,2026-04-19 11:34:47,"[Action, Drama, Romance]",False,...,1,0,0.00,0.000000,0,0,0,0,0,0
1,U02217,CT000222,C0282,Sci-Fi,mini,5.5,68,2025-10-29 22:59:01,[Sci-Fi],True,...,1,1,3.01,0.547273,0,0,0,0,0,1
2,U02802,CT005101,C0142,Action,image,160.2,40,2026-04-04 20:13:39,"[Fantasy, Action, Romance, Documentary]",True,...,1,1,80.08,0.499875,1,1,0,0,0,1
3,U02792,CT003164,C0029,Documentary,mini,47.8,91,2026-02-19 17:06:10,"[Sci-Fi, Fantasy]",False,...,1,0,0.00,0.000000,0,0,0,0,0,0
4,U00227,CT006055,C0400,Drama,image,30.2,18,2025-11-29 18:57:20,[Romance],False,...,1,0,0.00,0.000000,0,0,0,0,0,0


In [219]:
columns_to_remove = [
    "preferred_genres",
    "engagement_probability",
    "activity_score"
]

interactions = interactions.drop(
    columns=columns_to_remove
)

print(interactions.shape)

(250000, 22)


In [220]:
RAW_DATA_PATH = Path("../data/raw")
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)

interactions.to_csv(
    RAW_DATA_PATH / "interactions.csv",
    index=False
)

print("Interactions dataset saved successfully.")

Interactions dataset saved successfully.
